# Python, part 0 · Open the Box

**Goal:** see what the one line you ran in course 1 really does, and find out it's a short list of named steps that you can run one at a time.

There's no new Python to learn today. You'll run code, read it, and watch what it does. Parts 1 to 5 then teach you to write it.

**How to use this notebook** (the same as course 1)
- Start the lab cluster first (**Compute**; a cold start takes 5 minutes or more), then attach this notebook to it.
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = "P0"

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
import pandas as pd
from lnl import check, cleaning, domains, inside, pipeline
from lnl.check import ___

## 1. Remember this?
In course 1, session 2, one line took the file from a mess to clean data you could trust. Here it is again.

`raw` is the file exactly as the source system wrote it, every value as text. `clean` is what session 2 made of it.

In [ ]:
raw = pipeline.load_raw()
clean = cleaning.session2(raw)
inside.side_by_side(raw, clean)

That's one record, the one session 2 changed most, shown before and after. Text is in quotes so you can see the stray spaces.

Look at what changed:
- blanks, spaces and "don't know" codes like `N/A` became real empty values;
- the IDs became numbers;
- the original values were kept, and new columns at the bottom hold the cleaned versions: a matching copy of each name (`CAStillO` becomes `CASTILLO`) and the coded values mapped to standard ones.

Now picture doing that by hand for 100,000 records, every time a new file arrives. That's the value of the code. Today we open it up and look inside.

## 2. What's inside the one line
`inside.show` prints a piece of Python's own code, with line numbers. Here is everything `session2` is:

In [ ]:
inside.show(cleaning.session2)

Read it like a recipe:

| Line | Python | In plain English |
| --- | --- | --- |
| 1 | `def session2(df):` | Here is a recipe called `session2`. Give it a table, and inside the recipe the table is called `df`. |
| 2 | `"""Everything taught ..."""` | A note saying what the recipe is for. Python ignores it; people read it. |
| 3 | `df = strip_and_null(df)` | Do the step `strip_and_null` to the table, and keep the result under the same name. |
| 4 to 7 | `df = ...(df)` | The same again, one step per line. |
| 8 | `from .domains import ...` | Fetch one more step that lives in another file. |
| 9 | `return ...(df)` | Do that last step and hand the finished table back. |

So the whole of session 2 is a list of named steps, run in order. `inside.recipe` pulls the list out:

In [ ]:
inside.recipe(cleaning.session2)

Six steps. That's all the one line ever was.

## 3. Run the steps one at a time
Each step takes a table and hands back a new one. So you can run them yourself, one per cell, and look at the table in between.

After each step, `inside.report(before, after)` says what changed.

**Step 1 · `strip_and_null`:** trim spaces from both ends of every value, then turn what's left empty into a real empty value (a *null*).

In [ ]:
step1 = cleaning.strip_and_null(raw)
inside.report(raw, step1)

🧪 **Your turn #1.** How many values became empty (null) in step 1?

Read it off the report above: it's the number after the **+** sign. Type it **without the comma**. Python reads `1,234` as two numbers.

In [ ]:
answer_1 = ___      # a number from the report above, like 1234
check.p0_ex1(answer_1)

**Step 2 · `set_record_id`:** prove every offender number is a real, unique ID, then use it to label the rows.

In [ ]:
step2 = cleaning.set_record_id(step1)
inside.report(step1, step2)

**Step 3 · `apply_null_tokens`:** the codes the source system uses for "don't know" (`N/A`, `UNK`, `99:99`...) become real empty values too. The list of codes comes from the data dictionary.

In [ ]:
step3 = cleaning.apply_null_tokens(step2)
inside.report(step2, step3)

🧪 **Your turn #2.** How many more values became empty in step 3?

In [ ]:
answer_2 = ___      # a number from the report above, without the comma
check.p0_ex2(answer_2)

**Step 4 · `repair_text_columns`:** fix text damage, such as broken accents and web leftovers like `&nbsp;`.

In [ ]:
step4 = cleaning.repair_text_columns(step3)
inside.report(step3, step4)

**Step 5 · `standardize_names`:** add a matching copy of each name (capitals, accents folded), so `GARCÍA` and `Garcia` match.

In [ ]:
step5 = cleaning.standardize_names(step4)
inside.report(step4, step5)

**Step 6 · `standardize_domains`:** map coded values (race, religion, gang affiliation, offense, violence, unit) through the crosswalk files.

In [ ]:
step6 = domains.standardize_domains(step5)
inside.report(step5, step6)

🧪 **Your turn #3.** Which step added the three name columns that end in `_std`?

Type the step's name **in quotes**, like `"strip_and_null"`. Quotes tell Python it's text.

In [ ]:
answer_3 = ___      # a step's name, in quotes
check.p0_ex3(answer_3)

### Did the six steps give the same result as the one line?

In [ ]:
inside.same_table(step6, clean)

Six steps run by hand gave the same table as the one line, cell for cell. That's how you'll check your own code in the parts ahead: run it, then compare.

It's also how you find a problem. When a number looks wrong, run the steps one at a time and see which step changed it.

## 4. Open one step
Every step is a recipe too. Here is the first one, `strip_and_null`:

In [ ]:
inside.show(cleaning.strip_and_null)

| Python | In plain English |
| --- | --- |
| `out = df.copy()` | Work on a copy, so the original table is never changed. |
| `for col in text_columns(out):` | For every column that holds text, do the two indented lines below. |
| `stripped = out[col].str.strip()` | Trim spaces and tabs from both ends of every value in the column. |
| `out[col] = stripped.mask(stripped == "")` | Where nothing is left, put a real empty value instead. |
| `return out` | Hand the cleaned copy back. |

That's four ideas: a copy, a loop, a text method and a condition. Parts 1 to 3 teach exactly those. In part 3 you'll write `strip_and_null` yourself, and the check will compare your version with this one.

## Wrap-up
- Course 1's cleaning is Python you can read. A function is a named recipe, and session 2 is six named steps run in order.
- Each step takes a table and hands one back, so you can run the steps one at a time and look in between. That's also how you track down a wrong number.
- `inside.show(...)` opens any course function. Try it on a step from session 3: `from lnl import dates`, then `inside.show(dates.session3)`.

**Next, part 1 · Strings:** one messy name, the text methods that fix it, and your first function.

## Answers

**#1**
```python
answer_1 = 77734
```
**#2**
```python
answer_2 = 6122
```
**#3**
```python
answer_3 = "standardize_names"
```